# 매출 YoY vs 수출 HS코드 YoY 상관관계 분석 (읽기 전용)

DB에 어떤 것도 쓰지 않습니다 (SELECT만 사용). 반복적인 락/커넥션 문제를 원천 차단하기 위해
백필/저장 로직을 전부 제거하고, 이미 있는 데이터만으로 분석합니다.

## 처리 순서

### ① 매출 데이터
1. `US_IS_from_FMP`에서 티커별 분기 매출 로드
2. 최근 시점부터 거꾸로 봐서 **연속 20분기 이상** 인 티커만 채택
3. 분기 주기 3그룹으로 분리:
   - 표준 그룹: 1,2,3 / 4,5,6 / 7,8,9 / 10,11,12월
   - 그룹B: 2,3,4 / 5,6,7 / 8,9,10 / 11,12,1(다음해)월
   - 그룹C: 3,4,5 / 6,7,8 / 9,10,11 / 12,1(다음해),2(다음해)월
4. 그룹별로 티커마다 YoY 매출성장률 계산

### ② 수출 HS코드 데이터
1. 월별 실측 데이터 로드, **연속 60개월 이상**인 HS코드만 채택
2. 월별 패널에 `rolling(3, min_periods=3).sum()` 적용
   → 이 한 번의 연산으로 위 3가지 분기 패턴 전부에 대해 정확한 분기합계를 제공한다
   (연도 걸침 케이스 포함 5가지 시나리오로 직접 검증 완료)
3. 그 결과에 `.pct_change(12)` 적용 → HS코드별 분기 YoY

### ③ 상관계수 계산
각 그룹의 각 티커에 대해, **그 티커가 실제로 리포트한 날짜 그대로** HS YoY 패널에서
값을 조회합니다 (그룹 내 모든 티커가 같은 분기말 캘린더에 맞춰 리포트하므로 날짜가 정확히
일치 — 별도 근사 매칭이 필요 없습니다). 티커 1명당 전체 HS코드를 `DataFrame.corrwith()`로
한 번에 벡터화 계산합니다.

## 조회 인터페이스
- `ticker` 입력 → 상관계수 높은 순 HS 코드 n개
- `hs_code` 입력 → 상관계수 높은 순 ticker n개


In [1]:

# -*- coding: utf-8 -*-
from __future__ import annotations
import os, sys
from pathlib import Path


def _find_project_root(module_name="DATA", max_up=6):
    here = Path.cwd()
    for base in [here, *list(here.parents)[:max_up]]:
        if (base / module_name).is_dir():
            return base
    return None


try:
    from DATA.stock_invest_function import *
except ModuleNotFoundError:
    _root = _find_project_root("DATA")
    if _root is None:
        raise ModuleNotFoundError("DATA 패키지를 찾을 수 없습니다. 프로젝트 루트에서 실행해 주세요.")
    sys.path.insert(0, str(_root))
    from DATA.stock_invest_function import *
    print(f"[경로 자동 보정] DATA 모듈을 {_root} 에서 찾아 sys.path에 추가했습니다.")

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings("ignore")

db_info = {"host": get_db_host(), "port": 3307, "user": "stox7412",
           "password": "Apt106503!~", "database": "investar"}


def make_engine(db_info):
    url = (f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
           f"@{db_info['host']}:{int(db_info['port'])}/{db_info['database']}?charset=utf8mb4")
    return create_engine(url, pool_pre_ping=True, pool_recycle=1800,
                          connect_args={"connect_timeout": 10, "read_timeout": 60, "write_timeout": 60})


engine = make_engine(db_info)
TABLE_REVENUE = "US_IS_from_FMP"
TABLE_EXPORT_MONTHLY = "us_trade_export_monthly_with_forecast"
TABLE_IMPORT_MONTHLY = "us_trade_import_monthly_with_forecast"
DIRECTION = "export"   # 이 노트북 전용: "export" 또는 "import"

# 이 노트북은 SELECT 만 사용합니다. INSERT/UPDATE/DELETE/ALTER 없음.


In [2]:

# ============================================================
# 공통 헬퍼
# ============================================================

def _month_end(ts):
    return pd.Timestamp(ts) + pd.offsets.MonthEnd(0)


def get_trailing_consecutive_run(dates, tol_days=100):
    """'최근 시점부터 거꾸로' 봤을 때, 간격이 tol_days 이내로 이어지는 연속 구간만 추출."""
    dates = sorted(pd.to_datetime(d) for d in dates)
    if not dates:
        return []
    run = [dates[-1]]
    for i in range(len(dates) - 2, -1, -1):
        if (run[-1] - dates[i]).days <= tol_days:
            run.append(dates[i])
        else:
            break
    return sorted(run)


def compute_revenue_yoy(series: pd.Series, tol_days: int = 45) -> pd.Series:
    """기업 매출 시계열의 YoY. 1년 전 가장 가까운(±tol_days) 리포트와 비교."""
    series = series.sort_index()
    idx = series.index
    target_dates = idx - pd.DateOffset(years=1)
    base_vals = series.reindex(target_dates, method="nearest", tolerance=pd.Timedelta(days=tol_days))
    base_vals.index = idx
    with np.errstate(invalid="ignore", divide="ignore"):
        yoy = (series.values / base_vals.values - 1.0) * 100.0
    return pd.Series(yoy, index=idx).replace([np.inf, -np.inf], np.nan).dropna()


def load_trade_monthly_long(engine, direction="import"):
    if direction == "import":
        sql = text(f"""
            SELECT hs_code_6d AS hs_code, date, impDlr AS value
            FROM {TABLE_IMPORT_MONTHLY}
            WHERE forecast_flag = 0 AND impDlr IS NOT NULL
        """)
    elif direction == "export":
        with engine.connect() as conn:
            cols = [r[0] for r in conn.execute(text(f"SHOW COLUMNS FROM {TABLE_EXPORT_MONTHLY}")).fetchall()]
        vcol = "created_at" if "created_at" in cols else ("input_date" if "input_date" in cols else None)
        vcol_join = f"""
            JOIN (SELECT hs_code, MAX({vcol}) AS mx FROM {TABLE_EXPORT_MONTHLY} GROUP BY hs_code) l
              ON t.hs_code = l.hs_code AND t.{vcol} = l.mx
        """ if vcol else ""
        sql = text(f"""
            SELECT t.hs_code AS hs_code, t.date_month_end AS date, t.expDlr AS value
            FROM {TABLE_EXPORT_MONTHLY} t
            {vcol_join}
            WHERE t.is_forecast = 0 AND t.expDlr IS NOT NULL
        """)
    else:
        raise ValueError("direction must be 'import' or 'export'")

    with engine.connect() as conn:
        rows = conn.execute(sql).fetchall()
    df = pd.DataFrame(rows, columns=["hs_code", "date", "value"])
    df["date"] = pd.to_datetime(df["date"]).apply(_month_end)
    df["hs_code"] = df["hs_code"].astype(str).str.zfill(6)
    return df


def get_top_hscodes_for_ticker(correlation_df, ticker, n=10, min_abs_corr=None):
    df = correlation_df[correlation_df["ticker"] == ticker].copy()
    if min_abs_corr:
        df = df[df["correlation"].abs() >= min_abs_corr]
    df["abs_correlation"] = df["correlation"].abs()
    return (df.sort_values("abs_correlation", ascending=False)
              .drop(columns="abs_correlation").head(n).reset_index(drop=True))


def get_top_tickers_for_hscode(correlation_df, hs_code, n=10, min_abs_corr=None):
    hs_code = str(hs_code).zfill(6)
    df = correlation_df[correlation_df["hs_code"] == hs_code].copy()
    if min_abs_corr:
        df = df[df["correlation"].abs() >= min_abs_corr]
    df["abs_correlation"] = df["correlation"].abs()
    return (df.sort_values("abs_correlation", ascending=False)
              .drop(columns="abs_correlation").head(n).reset_index(drop=True))


## ① 매출 로드 + 연속 20분기 필터 + 분기주기 3그룹 분리 + YoY

In [3]:

MIN_QUARTERS = 20
QUARTER_TOL_DAYS = 100

sql = text(f"""
    SELECT ticker, date, value AS revenue
    FROM {TABLE_REVENUE}
    WHERE item = 'revenue' AND period IN ('Q1','Q2','Q3','Q4')
      AND value IS NOT NULL AND value > 0
    ORDER BY ticker, date
""")
with engine.connect() as conn:
    rows = conn.execute(sql).fetchall()
revenue_raw = pd.DataFrame(rows, columns=["ticker", "date", "revenue"])
revenue_raw["date"] = pd.to_datetime(revenue_raw["date"]).apply(_month_end)
revenue_raw = revenue_raw.drop_duplicates(subset=["ticker", "date"], keep="last")
print(f"[매출] 원본: 티커 {revenue_raw['ticker'].nunique():,}개 / 레코드 {len(revenue_raw):,}건")

kept_rows, n_qualified = [], 0
for ticker, g in revenue_raw.groupby("ticker"):
    run_dates = get_trailing_consecutive_run(g["date"].tolist(), tol_days=QUARTER_TOL_DAYS)
    if len(run_dates) >= MIN_QUARTERS:
        n_qualified += 1
        kept_rows.append(g[g["date"].isin(run_dates)])
revenue_filtered = pd.concat(kept_rows, ignore_index=True) if kept_rows else pd.DataFrame(columns=revenue_raw.columns)
print(f"[매출] 연속 {MIN_QUARTERS}분기 이상 만족 티커: {n_qualified:,}개")

revenue_filtered = revenue_filtered.copy()
revenue_filtered["cycle_group"] = revenue_filtered["date"].dt.month % 3
group_labels = {0: "표준그룹(3,6,9,12월분기말)", 1: "그룹B(4,7,10,1월분기말)", 2: "그룹C(5,8,11,2월분기말)"}
# 참고: 위 매핑은 '분기말 월'로 그룹을 나눈 것 -> 실제 분기 구성 월은 (분기말-2, 분기말-1, 분기말)

revenue_yoy_by_group = {}
for g_id, label in group_labels.items():
    df_g = revenue_filtered[revenue_filtered["cycle_group"] == g_id]
    pivot = df_g.pivot_table(index="date", columns="ticker", values="revenue", aggfunc="last")
    yoy_rows = []
    for ticker in pivot.columns:
        s = pivot[ticker].dropna()
        yoy = compute_revenue_yoy(s)
        if not yoy.empty:
            tmp = yoy.reset_index(); tmp.columns = ["date", "revenue_yoy"]; tmp["ticker"] = ticker
            yoy_rows.append(tmp)
    revenue_yoy_by_group[label] = (
        pd.concat(yoy_rows, ignore_index=True)[["ticker", "date", "revenue_yoy"]]
        if yoy_rows else pd.DataFrame(columns=["ticker", "date", "revenue_yoy"])
    )
    print(f"  - {label}: 티커 {revenue_yoy_by_group[label]['ticker'].nunique():,}개, "
          f"{len(revenue_yoy_by_group[label]):,}건")


[매출] 원본: 티커 1,962개 / 레코드 13,397건
[매출] 연속 20분기 이상 만족 티커: 0개


AttributeError: Can only use .dt accessor with datetimelike values

## ② 수출 HS코드 로드 + 연속 60개월 필터 + 분기합산 + YoY

In [ ]:

MIN_MONTHS = 60
MONTH_TOL_DAYS = 35

trade_long = load_trade_monthly_long(engine, direction=DIRECTION)
n_total_hs = trade_long["hs_code"].nunique()

kept, n_qualified_hs = [], 0
for hs_code, g in trade_long.groupby("hs_code"):
    run_dates = get_trailing_consecutive_run(g["date"].tolist(), tol_days=MONTH_TOL_DAYS)
    if len(run_dates) >= MIN_MONTHS:
        n_qualified_hs += 1
        kept.append(g[g["date"].isin(run_dates)])

print(f"[{DIRECTION}] 연속 {MIN_MONTHS}개월 이상 만족 HS코드: {n_qualified_hs:,}개 (전체 {n_total_hs:,}개 중)")

trade_filtered = pd.concat(kept, ignore_index=True) if kept else pd.DataFrame(columns=trade_long.columns)
monthly_panel = trade_filtered.pivot_table(index="date", columns="hs_code", values="value", aggfunc="sum")

# 핵심: rolling(3).sum() 한 번으로 3가지 분기 패턴(표준/그룹B/그룹C) 전부의 분기합계를 커버
trailing_quarter_panel = monthly_panel.rolling(3, min_periods=3).sum()
trade_yoy_panel = trailing_quarter_panel.pct_change(12) * 100.0

print(f"[{DIRECTION}] HS코드 YoY 패널 shape: {trade_yoy_panel.shape}")


## ③ 그룹별 상관계수 계산 (벡터화)

In [ ]:

all_results = []

for group_label, rev_yoy_df in revenue_yoy_by_group.items():
    if rev_yoy_df.empty:
        continue
    rev_pivot = rev_yoy_df.pivot_table(index="date", columns="ticker", values="revenue_yoy", aggfunc="last")

    for ticker in rev_pivot.columns:
        rev_series = rev_pivot[ticker].dropna()
        # 이 티커가 실제로 리포트한 날짜 그대로 조회 (그룹 내 전체가 같은 분기말 캘린더라 정확히 일치)
        trade_aligned = trade_yoy_panel.reindex(rev_series.index)

        corr_vec = trade_aligned.corrwith(rev_series)
        valid_mask = trade_aligned.notna() & rev_series.notna().values[:, None]
        n_obs = valid_mask.sum(axis=0)

        keep_idx = n_obs[n_obs >= 4].index
        if len(keep_idx) == 0:
            continue

        out = pd.DataFrame({
            "ticker": ticker, "hs_code": keep_idx,
            "correlation": corr_vec.loc[keep_idx].values,
            "n_periods": n_obs.loc[keep_idx].values,
            "group": group_label,
        })
        all_results.append(out)

correlation_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
correlation_df = correlation_df.dropna(subset=["correlation"])
print(f"전체 상관계수 레코드: {len(correlation_df):,}건 "
      f"(티커 {correlation_df['ticker'].nunique() if len(correlation_df) else 0}개 x "
      f"HS코드 {correlation_df['hs_code'].nunique() if len(correlation_df) else 0}개)")

save_path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\US_trade_revenue_corr"
os.makedirs(save_path, exist_ok=True)
fp = os.path.join(save_path, f"revenue_{DIRECTION}_corr_readonly_{pd.Timestamp.today():%Y%m%d}.csv")
correlation_df.to_csv(fp, index=False, encoding="utf-8-sig")
print(f"[저장] {fp}")


## ④ 조회 예시

In [ ]:

ticker_result = get_top_hscodes_for_ticker(correlation_df, ticker="AAPL", n=15)
print("[AAPL] 상관계수 상위 HS 코드")
print(ticker_result.to_string(index=False))

print()
hs_result = get_top_tickers_for_hscode(correlation_df, hs_code="854232", n=15)
print("[854232] 상관계수 상위 ticker")
print(hs_result.to_string(index=False))
